### 1. Importing libraries

In [118]:
import numpy as np 
import pandas as pd 
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

In [119]:
df = pd.read_csv(r"C:\Users\umera\Desktop\Summer 2026\DataSets\uci-secom - uci-secom.csv")

In [120]:
df.head()

,Time,0,1,2,3,4,5,6,7,8,...,581,582,583,584,585,586,587,588,589,Pass/Fail
0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1


### 2. Assessing the data

In [88]:
df.dtypes


Time          object
0            float64
1            float64
2            float64
3            float64
              ...   
586          float64
587          float64
588          float64
589          float64
Pass/Fail      int64
Length: 592, dtype: object

In [89]:
df_data=df.drop('Time', axis=1)

In [90]:
df_data.isnull().sum().value_counts()

6       100
1        92
2        84
0        53
9        48
24       43
3        24
4        24
14       20
7        20
8        12
260      12
1018     12
273       8
51        8
1429      4
794       4
715       4
1341      4
12        4
10        4
949       4
5         3
Name: count, dtype: int64

In [91]:
df_data.fillna(df_data.mean(), inplace=True)

### 3. Data Preprocessing

In [93]:
X=df_data.drop('Pass/Fail', axis=1)
Y=df_data['Pass/Fail']
le = LabelEncoder()
Y=le.fit_transform(Y)
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [94]:
print(X_train.shape)
print(X_test.shape)


(1253, 590)
(314, 590)


### 4. Training and Testing the model before feature selection

In [95]:
log_reg=LogisticRegression(max_iter=1000)
log_reg.fit(X_train,Y_train)
Y_pred=log_reg.predict(X_test)
score=accuracy_score(Y_test,Y_pred)
print("Accuracy score of the model is: ",score)


Accuracy score of the model is:  0.9076433121019108


c:\Users\umera\miniconda3\envs\EDA\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Feature Selection using Filter Methods

### 1. Duplicate Features

In [96]:
duplicated_columns=X_train.columns[X_train.T.duplicated()]
X_train.drop(duplicated_columns, axis=1,inplace=True)
X_test.drop(duplicated_columns,axis=1,inplace=True)

print("Shape of the dataset after removing duplicated columns: ",X_train.shape)

Shape of the dataset after removing duplicated columns:  (1253, 478)


### 2. Variance Threshold

In [ ]:

VT=VarianceThreshold(threshold=0.05)
X_filtered=VT.fit(X_train)

In [98]:
sum(VT.get_support())

np.int64(287)

In [99]:
columns=X_train.columns[VT.get_support()]
columns

Index(['0', '1', '2', '3', '4', '6', '12', '14', '15', '16',
       ...
       '570', '571', '572', '573', '574', '576', '577', '581', '585', '589'],
      dtype='object', length=287)

In [100]:
X_train=VT.transform(X_train)
X_test=VT.transform(X_test)
X_train=pd.DataFrame(X_train)
X_test=pd.DataFrame(X_test)

In [101]:
print(X_train.shape)

(1253, 287)


### 3. Correlation Threshold

In [103]:
Y_train=pd.Series(Y_train)

In [104]:
print(X_train.shape)
print(Y_train.shape)

print(X_train.index[:10])
print(Y_train.index[:10])

(1253, 287)
(1253,)
RangeIndex(start=0, stop=10, step=1)
RangeIndex(start=0, stop=10, step=1)


In [105]:
correlation_matrix=X_train.corr()
columns=correlation_matrix.columns
columns_toDrop=[]
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1,len(correlation_matrix.columns)):
        if abs(correlation_matrix.loc[columns[i],columns[j]])>0.95:
            if abs(X_train[columns[i]].corr(Y_train)) > abs(X_train[columns[j]].corr(Y_train)):
                columns_toDrop.append(columns[j])
                continue
            columns_toDrop.append(columns[i])
    

In [106]:
columns_toDrop=set(columns_toDrop)
print(len(columns_toDrop))

94


In [107]:
X_train.drop(columns_toDrop,axis=1,inplace=True)
X_test.drop(columns_toDrop,axis=1,inplace=True)

In [108]:
print(f'X train Shape: {X_train.shape} \nX test Shape: {X_test.shape}')


X train Shape: (1253, 193) 
X test Shape: (314, 193)


### 4. ANOVA

In [ ]:

sel=SelectKBest(f_classif,k=100).fit(X_train,Y_train)
X_train.columns[sel.get_support()]


Index([  0,   7,  11,  12,  13,  14,  16,  17,  19,  20,  21,  25,  26,  28,
        32,  37,  39,  40,  41,  42,  43,  44,  45,  46,  47,  53,  56,  57,
        58,  59,  60,  61,  62,  63,  64,  65,  66,  70,  80,  85,  90,  91,
        96,  97,  98,  99, 100, 101, 102, 105, 111, 121, 122, 123, 134, 135,
       138, 139, 140, 148, 149, 152, 159, 161, 164, 176, 177, 181, 184, 189,
       191, 192, 195, 197, 204, 208, 210, 213, 218, 219, 221, 224, 225, 226,
       229, 234, 235, 241, 242, 246, 248, 258, 259, 262, 266, 270, 271, 272,
       276, 280],
      dtype='int64')

In [110]:
columns = X_train.columns[sel.get_support()]
X_train=sel.transform(X_train)
X_test=sel.transform(X_test)
X_train=pd.DataFrame(X_train, columns=columns)
X_test=pd.DataFrame(X_test, columns=columns)



In [113]:
X_train.shape
X_test.shape

(314, 100)

## Checking the model performance after feature selection

In [116]:
log_reg=LogisticRegression(max_iter=1000)
log_reg.fit(X_train,Y_train)
Y_pred=log_reg.predict(X_test)
acc=accuracy_score(Y_pred,Y_test)
print('The Accuracy score of our model after Selecting 100 best features',acc)

The Accuracy score of our model after Selecting 100 best features 0.910828025477707


c:\Users\umera\miniconda3\envs\EDA\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Summary and Conclusion

In this notebook, we explored various filter methods for feature selection. We discussed the importance of selecting relevant features to improve model performance and reduce overfitting. The methods covered include duplicate feature removal, variance thresholding, correlation thresholding, and ANOVA. Each method has its own strengths and is suitable for different scenarios. The choice of method depends on the specific characteristics of the dataset and the goals of the machine learning task.We witnessed before and after feature selection model performance, our model's accuracy improved from 0.90 to 0.91 That mean the results are better after feature selection. In conclusion, feature selection is a crucial step in the machine learning pipeline that can lead to more efficient and effective models.